# Code

In [ ]:
# @title Setup Hugging Face Token
import os
from pathlib import Path

# Read token from environment first; fallback to optional local file.
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    token_file = Path("./.hf_token")
    if token_file.exists():
        hf_token = token_file.read_text(encoding="utf-8").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
else:
    print("HF_TOKEN not set. Model downloads may fail for gated repos.")

In [ ]:
# @title Setup the logger
%load_ext autoreload
%autoreload 2
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Keep notebook reruns idempotent and avoid duplicated stream handlers.
if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    ch.setFormatter(formatter)
    logger.addHandler(ch)

logging.info("The logger is setup")

In [ ]:
# @title Install libraries
# if first time running, uncomment the following lines to install the required libraries

# !pip install retriv
# !pip install datasets
# !pip install vllm==0.6.2
# !pip install jieba
# !pip install rank_bm25

In [ ]:
# @title Task configurations
tasks_info = {
    'binary':
        {
            'filename': 'balanced_binary_data.csv',
            'label_names': ['benign', 'illicit'],
            'positive_label': 'illicit',
            'prompt_prefix': f"""You will act as a content detector to decide if a given text is illicit promotion, i.e., involving the promotion of illicit goods and services.
If it is, label it as 'illicit', otherwise, label it as 'benign'.
First, you will be given a set of message examples alongwith their labels. Then, you will answer the category of the last message from the following options:
- 'benign': Content that does not contain illicit promotion
- 'illicit': Content that contain illicit promotion
Return only the name of the category, and nothing else."""
            },
    'binary_generalization':
    {
            'filename': 'balanced_category_data.csv',
            'label_names': ['benign', 'illicit'],
            'positive_label': 'illicit',
            'prompt_prefix': f"You will act as a content detector.\nFirst, you will be given a set of message examples along with their labels.\nThen, you will answer the category of the last message from the following options: 'benign', 'illicit'.\nReturn only the name of the category, and nothing else."
    },
    'category':
        {
            'filename': 'balanced_category_data.csv',
            'label_names': ['porn', 'gambling', 'drug', 'data-theft', 'money-laundry', 'counterfeit', 'advertisement', 'weapon', 'others', 'surrogacy', 'fraud', 'hacking', 'benign'],
            'prompt_prefix': "You will act as a content detector.\nFirst, you will be given a set of message examples along with their labels.\nThen, you will answer the category of the last message from the following options: 'porn', 'surrogacy', 'gambling', 'drug', 'data-theft', 'money-laundry', 'counterfeit', 'advertisement', 'weapon', 'fraud', 'hacking', 'others', 'benign'.\nReturn only the name of the category, and nothing else."
        },
    'category_sub':
        {
            'filename': 'balanced_category_data.csv',
            'label_names': ['porn', 'gambling', 'drug', 'counterfeit'],
            'prompt_prefix': "You will act as a content detector.\nFirst, you will be given a set of message examples along with their labels.\nThen, you will answer the category of the last message from the following options: 'porn', 'gambling', 'drug', 'counterfeit'.\nReturn only the name of the category, and nothing else."
        },
    'category_detailed':
    {
            'filename': 'balanced_category_data.csv',
            'label_names': ['porn', 'gambling', 'drug', 'data-theft', 'money-laundry', 'counterfeit', 'advertisement', 'weapon', 'others', 'surrogacy', 'fraud', 'hacking', 'benign'],
            'prompt_prefix':"""You will act as a content detector.
First, you will be given a set of message examples along with their labels. Then, you will answer the category of the last message from the following options:
- 'benign': Content that does not contain illicit promotion
- 'porn': Content that is sexually explicit or pornographic in nature
- 'surrogacy': Content related to surrogate mothers and gestational surrogacy
- 'gambling': Content related to gambling or betting activities
- 'drug': Content related to illegal drug use, sales, or promotions
- 'data-theft': Content involving the theft or illegal use of data, identity theft, or similar activities
- 'money-laundry': Content involving the promotion or recruitment for money laundering activities
- 'counterfeit': Content related to fake goods, forged certificates or false accounts
- 'advertisement': Content related to illegal marketing and black hat SEO
- 'weapon': Content related to weapons, including sales, manufacturing, or usage
- 'fraud': Content related to fraudulent activities and scams
- 'hacking': Content related to hacking, cybersecurity threats, or development of unlawful programs
- 'others': Content that does not fit into any of the above categories
Return only the name of the category, and nothing else."""
    },
}

model_name_id_map = {
    'llama3': 'meta-llama/Meta-Llama-3-8B-Instruct',
    'llama3.1': 'meta-llama/Meta-Llama-3.1-8B-Instruct',
    'llama3.1-non-instruct': 'meta-llama/Llama-3.1-8B',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.2',
    'mistral-non-instruct': 'mistralai/Mistral-7B-v0.3', # no 0.2-non-instruct official version found
    'phi3mini': 'microsoft/Phi-3-mini-128k-instruct',
    'phi3small': 'microsoft/Phi-3-small-128k-instruct',
    'gemma': 'google/gemma-2b-it',
    'qwen': 'Qwen/Qwen2.5-7B-Instruct',
}
logging.info("Task configurations are defined.")

In [ ]:
# @title Class Metrics
from sklearn import metrics
from typing import Dict, List


class Metrics:
    @staticmethod
    def cal_multiclass_metrics(
        y_true,
        y_pred,
        labels,
    ) -> Dict[str, Dict[str, float]]:
        precision = metrics.precision_score(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
        recall = metrics.recall_score(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
        f1 = metrics.f1_score(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
        accuracy = metrics.accuracy_score(y_true, y_pred)

        macro_precision = metrics.precision_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        )
        macro_recall = metrics.recall_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        )
        macro_f1 = metrics.f1_score(
            y_true, y_pred, labels=labels, average="macro", zero_division=0
        )

        metrics_dict = {
            "overall": {
                "precision": macro_precision,
                "recall": macro_recall,
                "f1": macro_f1,
                "accuracy": accuracy,
            }
        }

        metrics_dict.update(
            {
                label: {
                    "precision": float(precision[i]),
                    "recall": float(recall[i]),
                    "f1": float(f1[i]),
                }
                for i, label in enumerate(labels)
            }
        )
        return metrics_dict

    @staticmethod
    def cal_binary_metrics(
        y_true,
        y_pred,
        labels: List[str],
        pos_label,
    ) -> Dict[str, float]:
        labels = list(labels)
        if pos_label not in labels or len(labels) != 2:
            raise ValueError("Binary metrics require exactly 2 labels including pos_label")

        # Force label order: [positive, negative] for stable confusion matrix unpacking.
        neg_label = labels[0] if labels[1] == pos_label else labels[1]
        ordered_labels = [pos_label, neg_label]

        cm = metrics.confusion_matrix(y_true, y_pred, labels=ordered_labels)
        tp, fn = cm[0, 0], cm[0, 1]
        fp, tn = cm[1, 0], cm[1, 1]

        def safe_div(num, den):
            return float(num / den) if den else 0.0

        return {
            "precision": safe_div(tp, tp + fp),
            "recall": safe_div(tp, tp + fn),
            "f1": safe_div(2 * tp, 2 * tp + fp + fn),
            "fpr": safe_div(fp, fp + tn),
            "accuracy": safe_div(tp + tn, tp + tn + fp + fn),
            "pos_label": pos_label,
        }


y_true = ["good", "bad", "good", "bad", "good"]
y_pred = ["bad", "bad", "good", "bad", "good"]
labels = ["bad", "good"]
print(Metrics.cal_multiclass_metrics(y_true, y_pred, labels))


y_true = ["good", "bad", "good", "bad"]
y_pred = ["bad", "bad", "good", "bad"]
labels = ["bad", "good"]
print(Metrics.cal_binary_metrics(y_true, y_pred, labels, "bad"))


In [ ]:
# @title Function log_resources
import time

import psutil
import torch


def log_resources(start_time):
    process = psutil.Process()
    memory_info = process.memory_info()
    cpu_memory_usage = memory_info.rss / 1024 ** 2

    if torch.cuda.is_available():
        gpu_memory_usage = torch.cuda.memory_allocated() / 1024 ** 2
    else:
        gpu_memory_usage = 0

    total_time = time.time() - start_time

    logging.info(
        f"Total time: {total_time:.2f} s | CPU memory used: {cpu_memory_usage:.2f} MB | GPU memory used: {gpu_memory_usage:.2f} MB"
    )

# start_time = time.time()
# log_resources(start_time)

In [ ]:
# @title Class InContextLearner
import os
import random
import logging
from typing import List

import jieba
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from retriv import DenseRetriever
from vllm import LLM, SamplingParams

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
os.environ["VLLM_ALLOW_DEPRECATED_BEAM_SEARCH"] = "1"


class InContextLearner:
    def __init__(
        self,
        model_name: str,
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        model: LLM = None,
    ) -> None:
        self.model_name = model_name
        self.train_df = train_df
        self.test_df = test_df
        self.model = model

    def generate_prompts(
        self,
        n_shots: int,
        retrieval: str,
        prompt_prefix: str,
        query_prefix: str = "Query",
        answer_prefix: str = "Answer",
        random_seed: int = 42,
        has_demos_label: bool = True,
        shot_order: str = "fixed",
        shot_label_order: List[str] = None,
        first_shot_label: str = None,
        last_shot_label: str = None,
        needle_in_haystack: bool = False,
        generalization_label: str = None,
        generalization_label_mode: str = None,
    ) -> List[str]:
        prompts = []
        rng = random.Random(random_seed)

        def create_prompt(demonstrations, query):
            demos_local = list(demonstrations)

            if shot_label_order is not None:
                demonstrations_labels = [demo[1] for demo in demos_local]
                assert set(demonstrations_labels).issubset(
                    set(shot_label_order)
                ), "shot_label_order must contain all the labels in the demonstrations"
                demos_local.sort(key=lambda x: shot_label_order.index(x[1]))

            if first_shot_label is not None:
                demonstrations_labels = [demo[1] for demo in demos_local]
                assert (
                    first_shot_label in demonstrations_labels
                ), "first_shot_label must be in the demonstrations"
                first_shot_demos = [demo for demo in demos_local if demo[1] == first_shot_label]
                remaining_demos = [demo for demo in demos_local if demo[1] != first_shot_label]
                demos_local = first_shot_demos + remaining_demos
            elif last_shot_label is not None:
                demonstrations_labels = [demo[1] for demo in demos_local]
                assert (
                    last_shot_label in demonstrations_labels
                ), "last_shot_label must be in the demonstrations"
                last_shot_demos = [demo for demo in demos_local if demo[1] == last_shot_label]
                remaining_demos = [demo for demo in demos_local if demo[1] != last_shot_label]
                demos_local = remaining_demos + last_shot_demos

            if shot_order == "random":
                rng.shuffle(demos_local)

            if has_demos_label:
                demos = "==\n".join(
                    [f"{query_prefix}: {demo}\n{answer_prefix}: {answer}\n" for demo, answer in demos_local]
                )
            else:
                demos = "==\n".join(
                    [f"{query_prefix}: {demo}\n" for demo, _ in demos_local]
                )
            query_str = f"{query_prefix}: {query}\n{answer_prefix}: "
            return f"{prompt_prefix}\n==\n{demos}==\n{query_str}"

        if generalization_label is not None:
            if generalization_label_mode == "zero":
                sampled_demos = self.train_df.sample(0, random_state=random_seed)
            elif generalization_label != "benign":
                sampled_demos = pd.concat(
                    [
                        self.train_df[self.train_df["label"] == "benign"].sample(28, random_state=random_seed),
                        self.train_df[self.train_df["label"] != generalization_label]
                        .groupby("label")
                        .apply(lambda x: x.sample(3, random_state=random_seed))
                        .reset_index(drop=True)
                        .assign(label="illicit"),
                    ]
                ).sample(frac=1, random_state=random_seed).reset_index(drop=True)
                if generalization_label_mode == "included":
                    sampled_demos = pd.concat(
                        [
                            sampled_demos,
                            self.train_df[self.train_df["label"] == generalization_label]
                            .sample(3, random_state=random_seed)
                            .reset_index(drop=True)
                            .assign(label="illicit"),
                        ]
                    )
            else:
                sampled_demos = pd.concat(
                    [
                        self.train_df[self.train_df["label"] != "benign"]
                        .sample(32, random_state=random_seed)
                        .assign(label="illicit"),
                    ]
                ).sample(frac=1, random_state=random_seed).reset_index(drop=True)
                if generalization_label_mode == "included":
                    sampled_demos = pd.concat(
                        [
                            sampled_demos,
                            self.train_df[self.train_df["label"] == "benign"]
                            .sample(32, random_state=random_seed)
                            .reset_index(drop=True),
                        ]
                    ).sample(frac=1, random_state=random_seed).reset_index(drop=True)

            for query in self.test_df["text"]:
                demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                prompts.append(create_prompt(demonstrations, query))

        elif first_shot_label is not None or last_shot_label is not None:
            sampled_demos = (
                self.train_df.groupby("label")
                .apply(lambda x: x.sample(n_shots // 3, random_state=random_seed))
                .reset_index(drop=True)
            )
            for query in self.test_df["text"]:
                demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                prompts.append(create_prompt(demonstrations, query))

        elif needle_in_haystack:
            sampled_demos = self.train_df.sample(max(0, n_shots - 1), random_state=random_seed)
            for query_text, query_label in self.test_df[["text", "label"]].values:
                demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                insert_pos = rng.randint(0, max(0, len(demonstrations)))
                demonstrations.insert(insert_pos, (query_text, query_label))
                prompts.append(create_prompt(demonstrations, query_text))

        else:
            if n_shots == 0:
                for query in self.test_df["text"]:
                    prompts.append(create_prompt([], query))
                return prompts

            if retrieval == "random":
                if shot_order == "fixed":
                    sampled_demos = self.train_df.sample(n_shots, random_state=random_seed)
                else:
                    sampled_demos = self.train_df.sample(n_shots, random_state=42)

                for query in self.test_df["text"]:
                    demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                    prompts.append(create_prompt(demonstrations, query))

            elif retrieval == "lexical":
                labels = self.train_df["label"].unique()
                bm25_models = {}
                label_frames = {label: self.train_df[self.train_df["label"] == label] for label in labels}

                for label in labels:
                    corpus = [row["text"] for _, row in label_frames[label].iterrows()]
                    tokenized_corpus = [jieba.lcut_for_search(doc) for doc in corpus]
                    bm25_models[label] = BM25Okapi(tokenized_corpus)

                per_label = max(1, n_shots // len(labels))
                for query in self.test_df["text"]:
                    tokenized_query = jieba.lcut_for_search(query)
                    sampled_demos = []

                    for label in labels:
                        doc_scores = bm25_models[label].get_scores(tokenized_query)
                        inds = np.argsort(doc_scores)[::-1][:per_label]
                        sampled = label_frames[label].iloc[inds]
                        sampled_demos.append(sampled)

                    sampled_demos = pd.concat(sampled_demos)
                    demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                    prompts.append(create_prompt(demonstrations, query))

            elif retrieval == "semantic":
                labels = self.train_df["label"].unique()
                st_retrievers = {}
                label_frames = {label: self.train_df[self.train_df["label"] == label] for label in labels}
                per_label = max(1, n_shots // len(labels))

                for label in labels:
                    collection = [
                        {"id": idx, "text": row["text"]}
                        for idx, row in label_frames[label].iterrows()
                    ]
                    st_retrievers[label] = DenseRetriever(
                        index_name=f"training-examples-{label}",
                        model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                        normalize=True,
                        max_length=128,
                        use_ann=False,
                    ).index(collection, use_gpu=True)

                for query in self.test_df["text"]:
                    sampled_demos = []

                    for label in labels:
                        retriever = st_retrievers[label]
                        retrieved = retriever.search(query=query, cutoff=per_label)
                        inds = [item["id"] for item in retrieved]
                        if len(inds) < per_label:
                            extra = label_frames[label].sample(per_label - len(inds), random_state=random_seed).index
                            inds.extend(extra)
                        sampled = label_frames[label].loc[inds]
                        sampled_demos.append(sampled)

                    sampled_demos = pd.concat(sampled_demos)
                    demonstrations = [(demo["text"], demo["label"]) for _, demo in sampled_demos.iterrows()]
                    prompts.append(create_prompt(demonstrations, query))

            else:
                raise AssertionError(f"Retrieval method {retrieval} is not supported")

        return prompts

    def predict(
        self,
        prompts: List[str],
        label_names: List[str],
    ):
        label_tokens = []
        tokenizer = self.model.get_tokenizer()
        for label_name in label_names:
            label_tokens.append(tokenizer.tokenize(label_name))

        max_tokens = max(len(label) for label in label_tokens)
        min_tokens = min(len(label) for label in label_tokens)
        n_candidates = min(50, max(20, len(label_names) * 4))

        sampling_params = SamplingParams(
            temperature=0.0,
            use_beam_search=True,
            n=n_candidates,
            top_p=1.0,
            top_k=-1,
            max_tokens=max_tokens,
            min_tokens=min_tokens,
            logprobs=10,
        )

        outputs = self.model.generate(prompts, sampling_params)
        return outputs

    def evaluate(
        self,
        outputs,
        label_names: List[str],
        pos_label: str = "illicit",
    ):
        def calc_metrics(output_texts: List[str], test_df, label_names: List[str], pos_label: str):
            predicted_labels = []
            label_names_lower = [label.lower() for label in label_names]

            for output in output_texts:
                output_norm = (output or "").lower()
                matched_label = ""
                matched_index = len(output_norm)

                for label, label_lower in zip(label_names, label_names_lower):
                    index = output_norm.find(label_lower)
                    if index != -1 and index < matched_index:
                        matched_label = label
                        matched_index = index

                predicted_labels.append(matched_label)

            true_labels = test_df["label"]
            if len(label_names) == 2:
                metrics_result = Metrics.cal_binary_metrics(
                    true_labels, predicted_labels, label_names, pos_label
                )
                metrics_result["pos_label"] = pos_label
            else:
                metrics_result = Metrics.cal_multiclass_metrics(
                    true_labels, predicted_labels, label_names
                )

            predicted_labels = pd.Series(predicted_labels, index=test_df.index, name="predicted")
            save_state = pd.concat([predicted_labels, true_labels], axis=1)
            return metrics_result, save_state

        def calc_confidence(label_names, candidate_seqs_all):
            confidence_all = []
            label_names_lower = [label.lower() for label in label_names]
            label_to_idx = {label: i for i, label in enumerate(label_names_lower)}

            for candidate_seqs in candidate_seqs_all:
                scores = np.zeros(len(label_names), dtype=float)
                for seq in candidate_seqs:
                    output_label_token = (seq.text or "").strip().lower()
                    if output_label_token in label_to_idx:
                        scores[label_to_idx[output_label_token]] += np.exp(seq.cumulative_logprob)

                denom = scores.sum()
                if denom > 0:
                    probs = scores / denom
                else:
                    probs = np.zeros_like(scores)

                confidence_all.append({label: float(prob) for label, prob in zip(label_names, probs)})

            return confidence_all

        output_text_all = []
        candidate_seqs_all = []

        for output in outputs:
            if len(output.outputs) == 0:
                output_text_all.append("")
                candidate_seqs_all.append([])
            else:
                output_text_all.append(output.outputs[0].text)
                candidate_seqs_all.append(output.outputs)

        metrics, save_state = calc_metrics(
            output_text_all, self.test_df, label_names, pos_label
        )
        confidence_all = calc_confidence(label_names, candidate_seqs_all)
        return metrics, confidence_all


logging.info("It is done to define the ICL learner")

In [ ]:
# @title Function run_experiment

import json
import logging
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from vllm import LLM

# Cache dataset loading to avoid repeated CSV parsing in large experiment sweeps.
_DATASET_CACHE: Dict[str, pd.DataFrame] = {}


def _load_task_dataframe(dataset_path: str) -> pd.DataFrame:
    if dataset_path not in _DATASET_CACHE:
        dataset = load_dataset("csv", data_files=dataset_path)
        _DATASET_CACHE[dataset_path] = dataset["train"].to_pandas()
    return _DATASET_CACHE[dataset_path].copy()


def _resolve_model_name(model_name: str) -> str:
    return model_name_id_map.get(model_name, model_name)


def run_experiment(
    exp_name: str,
    task_name: str,
    model_name: str,
    model: LLM = None,
    data_dir: str = "./Data",
    output_dir: str = "./Result",
    overwrite: bool = False,
    data_random_seed: int = 42,
    exp_random_seed: int = 42,
    n_runs: int = 1,
    n_shots: int = 4,
    retrieval: str = "random",
    has_demos_label: bool = True,
    shot_order: str = "fixed",
    shot_label_order: List[str] = None,
    first_shot_label: str = None,
    last_shot_label: str = None,
    needle_in_haystack: bool = False,
    generalization_mode: str = "none",
    generalization_label_tuple: Tuple[str, str] = (None, None),
    label_map: dict = None,
):
    assert task_name in tasks_info.keys(), "Unsupported task"
    assert retrieval in {"random", "lexical", "semantic"}, "Unsupported retrieval method"
    assert n_runs >= 1, "n_runs must be >= 1"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    task_info = tasks_info[task_name].copy()
    model_name = _resolve_model_name(model_name)

    logging.info(f"* Starting {exp_name}_{n_shots}_{retrieval} with model {model_name}")

    dataset_path = str(Path(data_dir) / task_info["filename"])
    df = _load_task_dataframe(dataset_path)

    generalization_label, generalization_label_mode = generalization_label_tuple
    if generalization_label_tuple != (None, None):
        train_df, test_df = train_test_split(
            df, test_size=0.5, random_state=data_random_seed
        )
        test_df = test_df.drop(test_df[test_df["label"] != generalization_label].index)
        if generalization_label == "benign":
            test_df["label"] = "benign"
        else:
            test_df["label"] = "illicit"
    else:
        train_df, test_df = train_test_split(
            df, test_size=0.2, random_state=data_random_seed
        )
        if generalization_mode == "source_se2tw":
            train_df = train_df[train_df["source"] == "se"]
            test_df = test_df[test_df["source"] == "twitter"]
        elif generalization_mode == "source_tw2se":
            train_df = train_df[train_df["source"] == "twitter"]
            test_df = test_df[test_df["source"] == "se"]
        elif generalization_mode == "source_all2tw":
            test_df = test_df[test_df["source"] == "twitter"]
        elif generalization_mode == "source_all2se":
            test_df = test_df[test_df["source"] == "se"]
        elif generalization_mode == "source_tw2tw":
            train_df = train_df[train_df["source"] == "twitter"]
            test_df = test_df[test_df["source"] == "twitter"]
        elif generalization_mode == "source_se2se":
            train_df = train_df[train_df["source"] == "se"]
            test_df = test_df[test_df["source"] == "se"]

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    if label_map is not None:
        assert set(label_map.keys()) == set(
            task_info["label_names"]
        ), "Label mapping does not match the task label names"
        train_df["label"] = train_df["label"].map(label_map)
        test_df["label"] = test_df["label"].map(label_map)
        task_info["label_names"] = list(label_map.values())
        if task_name in {"binary", "binary_generalization"}:
            task_info["positive_label"] = label_map[task_info["positive_label"]]
        task_info[
            "prompt_prefix"
        ] = f"You will act as a content detector.\nFirst, you will be given a set of message examples along with their labels.\nThen, you will answer the category of the last message from the following options: {', '.join(task_info['label_names'])}.\nReturn only the name of the category, and nothing else."
        logging.info(f"* Label mapping is applied: {label_map}")

    logging.info(
        f"* Running on dataset {task_info['filename']}. Train Size = {len(train_df)}  Test Size = {len(test_df)}"
    )

    icl = InContextLearner(model_name, train_df, test_df, model=model)

    for run in range(n_runs):
        logging.info(f"--- run: {run} ---")
        random_seed = exp_random_seed + run
        result_prefix = f"{exp_name}_{task_name}_{model_name.replace('/', '+').replace('.', '-')}_r{random_seed}_n{n_shots}_r{retrieval}"

        outputs_file = output_path / f"{result_prefix}_outputs.pkl"
        if outputs_file.exists() and not overwrite:
            logging.info("Outputs are already there")
            continue

        try:
            prompts = icl.generate_prompts(
                n_shots=n_shots,
                retrieval=retrieval,
                prompt_prefix=task_info["prompt_prefix"],
                random_seed=random_seed,
                has_demos_label=has_demos_label,
                shot_order=shot_order,
                shot_label_order=shot_label_order,
                needle_in_haystack=needle_in_haystack,
                first_shot_label=first_shot_label,
                last_shot_label=last_shot_label,
                generalization_label=generalization_label,
                generalization_label_mode=generalization_label_mode,
            )
            outputs = icl.predict(prompts, task_info["label_names"])
        except AssertionError as e:
            logging.info(e)
            return

        with open(outputs_file, "wb") as f:
            pickle.dump(outputs, f)
        logging.info(f"* Raw outputs are saved: {outputs_file}")

        metrics, confidence = icl.evaluate(
            outputs,
            task_info["label_names"],
            pos_label=task_info["positive_label"] if "positive_label" in task_info else None,
        )
        logging.info(f"* Finish predicting. Peformance = {metrics}")

        confidence_to_save_df = pd.concat([test_df["text"], pd.DataFrame(confidence)], axis=1)
        confidence_file = output_path / f"{result_prefix}_confidence.csv"
        confidence_to_save_df.to_csv(confidence_file, index=False)
        logging.info(f"* Details of classification confidence are saved: {confidence_file}")

        result = {
            "exp": exp_name,
            "task": task_name,
            "model": model_name,
            "random_seed": random_seed,
            "n_shots": n_shots,
            "retrieval": retrieval,
            "metrics": metrics,
        }

        result_file = output_path / "results_all.json"
        with open(result_file, "a", encoding="utf-8") as file:
            file.write(json.dumps(result, ensure_ascii=False) + "\n")

        logging.info(f"* Experiment records file updated: {result_file}")


logging.info("Defining run_experiment is done")

# Execution

## Llama-3.1-8B-Instruct (128K)

### Load the model

In [ ]:
model_llama3_1 = LLM(model='meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True, tensor_parallel_size=2)

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary', 'category']
shots = [0, 2, 4, 8, 16, 32, 64, 128]
retrievals = ['random', 'lexical', 'semantic']

for task in tasks:
  for shot in shots:
    if shot == 0:
      run_experiment(exp_name='exp1', task_name=task, model_name='llama3.1', n_shots=shot, model=model_llama3_1, retrieval='random')
      continue
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1', task_name=task, model_name='llama3.1', n_shots=shot, retrieval=retrieval, model=model_llama3_1, n_runs=10)
      else:
        run_experiment(exp_name='exp1', task_name=task, model_name='llama3.1', n_shots=shot, retrieval=retrieval, model=model_llama3_1)

### Exp 1.2: Instruct vs Non-Instruct

This block checks whether instruction tuning improves binary detection quality under the same ICL pipeline.
We keep retrieval as `semantic` and compare model variants directly.

In [ ]:
# non-instruct version

model_llama3_1_noninstruct = LLM(model='meta-llama/Llama-3.1-8B', trust_remote_code=True, enable_prefix_caching=True, max_model_len=32768)
shots = [2, 4, 8, 16, 32, 64]
for shot in shots:
  run_experiment(exp_name='exp1', task_name='binary', model_name='llama3.1-non-instruct', n_shots=shot, model=model_llama3_1_noninstruct, retrieval='semantic')


### Exp 1.3: Demonstrations Without Labels

This ablation removes answer labels from demonstrations to measure how much explicit supervision contributes in-context.
The setup follows Exp 1 but sets `has_demos_label=False`.

In [ ]:
# w/o demonstrations label

tasks = ['binary']
shots = [32]
retrievals = ['random', 'semantic']

for task in tasks:
  for shot in shots:
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1_3_demos_label', task_name=task, model_name='llama3.1', n_shots=shot, retrieval=retrieval, model=model_llama3_1, n_runs=3, has_demos_label=False)
      else:
        run_experiment(exp_name='exp1_3_demos_label', task_name=task, model_name='llama3.1', n_shots=shot, retrieval=retrieval, model=model_llama3_1, has_demos_label=False)

### Exp 5: Context Length Sensitivity

Evaluate metric stability and compute cost as context size changes.
This helps identify practical context limits for deployment.

In [ ]:
context_lengths = [8192, 16384, 32768, 65536, 131072]

for context_length in context_lengths:
    model_llama3_1 = LLM(model='meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True, max_model_len=context_length, tensor_parallel_size=2)

    start_time = time.time()
    run_experiment(exp_name=f'exp5_contextlength_{context_length}', task_name='binary', model_name=f'llama_{context_length}', n_shots=32, model=model_llama3_1, retrieval='semantic')
    log_resources(start_time)

### Exp 7: Needle-in-Haystack Robustness

A sanity test where the query sample is inserted into demonstrations.
In principle, a robust ICL decision process should approach perfect recovery in this controlled setup.

In [ ]:
shots = [2, 4, 8, 16, 32, 64, 128]

for shot in shots:
    run_experiment(exp_name='exp7_haystack', task_name='binary', model_name='llama3.1', n_shots=shot, model=model_llama3_1, retrieval='random', needle_in_haystack=True)

## Mistral-7B-Instruct (32K)

### Load the model

In [ ]:
model_mistral = LLM(model='mistralai/Mistral-7B-Instruct-v0.2', trust_remote_code=True, tensor_parallel_size=2)

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary', 'category']
shots = [0, 2, 4, 8, 16, 32, 64, 128]
retrievals = ['random', 'lexical', 'semantic']
for task in tasks:
  for shot in shots:
    try:
      if shot == 0:
        run_experiment(exp_name='exp1', task_name=task, model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random')
        continue
      for retrieval in retrievals:
        if retrieval == 'random':
          run_experiment(exp_name='exp1', task_name=task, model_name='mistral', n_shots=shot, retrieval=retrieval, model=model_mistral, n_runs=10)
        else:
          run_experiment(exp_name='exp1', task_name=task, model_name='mistral', n_shots=shot, retrieval=retrieval, model=model_mistral)
    except AssertionError as e:
      print(e)
      continue
    except Exception as e:
      print(e)
      continue


### Exp 1.2: Instruct vs Non-Instruct

This block checks whether instruction tuning improves binary detection quality under the same ICL pipeline.
We keep retrieval as `semantic` and compare model variants directly.

Note: Since we cannot find the non-instruct version of `mistral-v0.2`, we use `mistral-v0.3` as a replacement.

In [ ]:
# @title instruct vs. non-instruct
model_mistral_instruct = LLM(model='mistralai/Mistral-7B-Instruct-v0.3', trust_remote_code=True, max_model_len=32768)
run_experiment(exp_name='exp1', task_name='binary', model_name='mistralai/Mistral-7B-Instruct-v0.3', n_shots=32, model=model_mistral_instruct, retrieval='semantic')

In [ ]:
model_mistral_noninstruct = LLM(model='mistralai/Mistral-7B-v0.3', trust_remote_code=True, max_model_len=32768)
run_experiment(exp_name='exp1', task_name='binary', model_name='mistralai/Mistral-7B-v0.3', n_shots=32, model=model_mistral_noninstruct, retrieval='semantic')

### Exp 1.3: Demonstrations Without Labels

This ablation removes answer labels from demonstrations to measure how much explicit supervision contributes in-context.
The setup follows Exp 1 but sets `has_demos_label=False`.

In [ ]:
# w/o demos label

tasks = ['binary', 'category']
shots = [32]
retrievals = ['random', 'semantic']

for task in tasks:
  for shot in shots:
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='mistral', n_shots=shot, retrieval=retrieval, model=model_mistral, n_runs=3, has_demos_label=False)
      else:
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='mistral', n_shots=shot, retrieval=retrieval, model=model_mistral, has_demos_label=False)

### Exp 2: Comparison with LoRA Fine-Tuning

This section compares ICL-only baselines against a LoRA-adapted checkpoint.
The goal is to quantify when few-shot prompting can match or trail lightweight fine-tuning.

In [ ]:
random_seed = 42
tasks = ['binary', 'category']
K_values = [280, 560, 1120, 2240, 4480]
epochs = [1, 2, 3]

for task in tasks:
    for K in K_values:
        for epoch in epochs:
            model_name = f'mistral_lora_model_{K}_r{random_seed}_e{epoch}'
            model_mistral_finetuned = LLM(model=f'./autodl-tmp/{model_name}', tensor_parallel_size=2)
            run_experiment(exp_name='exp2_finetuned', task_name=task, model_name=f'mistral_lora_model_{K}_e{epoch}', exp_random_seed=random_seed, n_shots=0, model=model_mistral_finetuned, retrieval='random')

### Exp 3: Shot Order Sensitivity

This block shuffles demonstration order across multiple runs.
It measures variance caused purely by permutation effects at fixed K and retrieval method.
The setup follows Exp 1 but sets `shot_order='random'`.

In [ ]:
tasks = ['binary', 'category']
shots = [4, 8, 16, 32, 64, 128]

for task in tasks:
    for shot in shots:
        run_experiment(exp_name='exp3_shotorder', task_name=task, model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', shot_order='random', n_runs=5)

### Exp 4: Label Order Effect

Demonstrations are arranged by label order (including first/last label controls).
This tests whether positional label bias influences downstream predictions.
The setup follows Exp 1 but sets `shot_label_order=[...]`.

In [ ]:
from itertools import permutations
labels = tasks_info['binary']['label_names']
labels_permutations = list(permutations(labels))
shots = [4, 8, 16, 32, 64, 128]

for shot in shots:
    for i, shot_label_order in enumerate(labels_permutations):
        run_experiment(exp_name=f'exp4_labelorder{i}', task_name='binary', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='semantic', shot_label_order=shot_label_order)

In [ ]:
shots = [32,64]
label_names = tasks_info['category_sub']['label_names']
for shot in shots:
    run_experiment(exp_name=f'exp4_labelorder_default', task_name='category_sub', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', first_shot_label='porn', shot_order='random', n_runs=1)
    for label in label_names:
        run_experiment(exp_name=f'exp4_labelorder_{label}_first', task_name='category_sub', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', first_shot_label=label, n_runs=1)
        run_experiment(exp_name=f'exp4_labelorder_{label}_last', task_name='category_sub', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', last_shot_label=label, n_runs=1)

### Exp 5: Context Length Sensitivity

Evaluate metric stability and compute cost as context size changes.
This helps identify practical context limits for deployment.

In [ ]:
context_lengths = [8192, 16384, 32768]

for context_length in context_lengths:
    model_mistral = LLM(model='mistralai/Mistral-7B-Instruct-v0.2', trust_remote_code=True, max_model_len=context_length, tensor_parallel_size=2)

    start_time = time.time()
    run_experiment(exp_name=f'exp5_contextlength_{context_length}', task_name='binary', model_name=f'mistral_{context_length}', n_shots=32, model=model_mistral, retrieval='semantic')
    log_resources(start_time)


### Exp 6: Generalization Across Sources and Categories

This part evaluates transfer under source shift (`se -> twitter`, `twitter -> se`) and category-held-out settings.
It reflects real-world robustness where train and test distributions differ.

In [ ]:
for shot in [0, 2, 4, 8, 16, 32, 64, 128]:
    run_experiment(exp_name=f'exp6_generalization_se2tw', task_name='binary', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='semantic', generalization_mode='source_se2tw')
    
    run_experiment(exp_name=f'exp6_generalization_tw2se', task_name='binary', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='semantic', generalization_mode='source_tw2se')

In [ ]:
# category generalization
shot = 64
labels = tasks_info['category']['label_names'].copy()
labels.remove('benign')
print(labels)

for label in labels:
    run_experiment(exp_name=f'exp6_generalization_{label}_zero', task_name='binary_generalization', model_name='mistral', n_shots=0, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'zero'))
    run_experiment(exp_name=f'exp6_generalization_{label}_included', task_name='binary_generalization', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'included'))
    run_experiment(exp_name=f'exp6_generalization_{label}_excluded', task_name='binary_generalization', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'excluded'))

In [ ]:
# category generalization add benign
shot = 64
label = 'benign'

run_experiment(exp_name=f'exp6_generalization_{label}_zero', task_name='binary_generalization', model_name='mistral', n_shots=0, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'zero'))
run_experiment(exp_name=f'exp6_generalization_{label}_included', task_name='binary_generalization', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'included'))
run_experiment(exp_name=f'exp6_generalization_{label}_excluded', task_name='binary_generalization', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', generalization_label_tuple=(label, 'excluded'))

### Exp 7: Needle-in-Haystack Robustness

A sanity test where the query sample is inserted into demonstrations.
In principle, a robust ICL decision process should approach perfect recovery in this controlled setup.

In [ ]:
tasks = ['binary', 'category']
shots = [2, 4, 8, 16, 32, 64, 128]

for task in tasks:
    for shot in shots:
        run_experiment(exp_name='exp7_haystack', task_name=task, model_name='mistral', n_shots=shot, model=model_mistral, retrieval='random', needle_in_haystack=True)

### Exp 10: Label Naming Effect

This section changes label surface forms (e.g., `benign/illicit` to `0/1`).
It verifies whether performance depends on semantic label wording versus decision boundary quality.

In [ ]:
for shot in [0, 2, 4, 8, 16, 32, 64, 128]:
    run_experiment(exp_name=f'exp10_labelname_01', task_name='binary', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='semantic', label_map={'benign': '0', 'illicit': '1'})
    run_experiment(exp_name=f'exp10_labelname_AB', task_name='binary', model_name='mistral', n_shots=shot, model=model_mistral, retrieval='semantic', label_map={'benign': 'A', 'illicit': 'B'})

In [ ]:
category_label_names = tasks_info['category']['label_names']
shots = [0, 2, 4, 8, 16, 32, 64, 128]

label_map_num = {label: str(i) for i, label in enumerate(category_label_names)}

label_map_alpha = {
    label: chr(ord('A') + i)
    for i, label in enumerate(category_label_names)
}

for shot in shots:
    run_experiment(
        exp_name='exp10_labelname_category_01',
        task_name='category',
        model_name='mistral',
        n_shots=shot,
        model=model_mistral,
        retrieval='semantic',
        label_map=label_map_num,
    )
    run_experiment(
        exp_name='exp10_labelname_category_AB',
        task_name='category',
        model_name='mistral',
        n_shots=shot,
        model=model_mistral,
        retrieval='semantic',
        label_map=label_map_alpha,
    )

## Phi-3-Small (128K)

### Load the model

In [ ]:
model_phi3small = LLM(model='microsoft/Phi-3-small-128k-instruct', trust_remote_code=True, 
                      tensor_parallel_size=2, 
                      enable_prefix_caching=False, 
                      enable_chunked_prefill=False,
                      enforce_eager=True,
                      max_model_len=65536)

# enable_prefix_caching, enable_chunked_prefill cannot be True!

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary', 'category']
shots = [0, 2, 4, 8, 16, 32, 64, 128]
retrievals = ['random', 'lexical', 'semantic']

for task in tasks:
  for shot in shots:
    if shot == 0:
      run_experiment(exp_name='exp1', task_name=task, model_name='phi3small', n_shots=shot, model=model_phi3small, retrieval='random')
      continue
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1', task_name=task, model_name='phi3small', n_shots=shot, retrieval=retrieval, model=model_phi3small, n_runs=10)
      else:
        run_experiment(exp_name='exp1', task_name=task, model_name='phi3small', n_shots=shot, retrieval=retrieval, model=model_phi3small)

### Exp 1.3: Demonstrations Without Labels

This ablation removes answer labels from demonstrations to measure how much explicit supervision contributes in-context.
The setup follows Exp 1 but sets `has_demos_label=False`.

In [ ]:
# w/o demos label

tasks = ['binary', 'category']
shots = [32]
retrievals = ['random', 'semantic']

for task in tasks:
  for shot in shots:
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='phi3small', n_shots=shot, retrieval=retrieval, model=model_phi3small, n_runs=3, has_demos_label=False)
      else:
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='phi3small', n_shots=shot, retrieval=retrieval, model=model_phi3small, has_demos_label=False)

### Exp 7: Needle-in-Haystack Robustness

A sanity test where the query sample is inserted into demonstrations.
In principle, a robust ICL decision process should approach perfect recovery in this controlled setup.

In [ ]:
shots = [2, 4, 8, 16, 32, 64, 128]

for shot in shots:
    run_experiment(exp_name='exp7_haystack', task_name='binary', model_name='phi3small', n_shots=shot, model=model_phi3small, retrieval='random', needle_in_haystack=True)

### Debug / Temporary Verification

This sandbox section is reserved for troubleshooting retrieval, prompt construction, and output parsing.
It is not part of the final benchmark table but documents intermediate checks for reproducibility.

In [ ]:
# Debug
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from retriv import SparseRetriever, DenseRetriever

data_dir = '/content/drive/MyDrive/LLM_ICL/Workspace/Data'
task_info = tasks_info['binary']

dataset = load_dataset('csv', data_files=f"{data_dir}/{task_info['filename']}")
df = dataset['train'].to_pandas()
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
print(f"* Train Size = {len(train_df)}  Test Size = {len(test_df)}")

collection = [{"id": idx, "text": row["text"]} for idx, row in train_df.iterrows()]

retriever = DenseRetriever(
    index_name="training-examples",
    model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    normalize=True,
    max_length=128,
    use_ann=False,
).index(collection, use_gpu=True)

In [ ]:
n_shots = 32
# Generate prompts
prompts = []

def create_prompt(demonstrations, query):
  demos = "==\n".join(
    [f"Query: {demo}\n" for demo, _ in demonstrations]
  )
  query_str = f"Query: {query}\nAnswer: "
  return f"{task_info['prompt_prefix']}\n==\n{demos}==\n{query_str}"

for query in test_df['text']:
  retrieved = retriever.search(
      query=query,
      cutoff=n_shots,
  )
  inds = [item['id'] for item in retrieved]
  if len(inds) < n_shots:
      inds.extend(train_df.sample(n_shots-len(inds), random_state=42).index)
  sampled_demos = train_df.loc[inds]
  demonstrations = [(demo['text'], demo['label']) for _, demo in sampled_demos.iterrows()]
  prompts.append(create_prompt(demonstrations, query))

In [ ]:
from vllm import SamplingParams, LLM

# Predict using the model and the prompts
model_phi3small = LLM(model='microsoft/Phi-3-small-128k-instruct', trust_remote_code=True, max_model_len=32768, enforce_eager=True)
label_tokens = []
tokenizer = model_phi3small.get_tokenizer()
for label_name in task_info['label_names']:
  label_tokens.append(tokenizer.tokenize(label_name))
max_tokens = max([len(label) for label in label_tokens])
min_tokens = min([len(label) for label in label_tokens])
sampling_params = SamplingParams(
    temperature=0.0,  # more deterministic
    use_beam_search=True, # maintain n candidate sequences
    n=50, # set n to cover the whole label_names space
    top_p=1.0,  # controls the cumulative probability of the top tokens to consider, set to 1 to consider all tokens
    top_k=-1,  # controls the number of top tokens to consider, set to -1 to consider all tokens
    max_tokens=max_tokens,
    min_tokens=min_tokens,
    logprobs=10,
)

# Return the predictions
outputs = model_phi3small.generate(prompts, sampling_params)

In [ ]:
from typing import List
import pandas as pd
import numpy as np
import os
import json

result_prefix = f"/content/drive/MyDrive/LLM_ICL/Workspace/Result/exp2_demos_label_binary_microsoft+Phi-3-small-128k-instruct_r42_n{n_shots}_rsemantic"

def evaluate(
    outputs,
    label_names: List[str],
    pos_label: str = 'illicit',
):

    def calc_metrics(outputs: List[str], test_df, label_names: List[str], pos_label: str):
      predicted_labels = []
      for output in outputs:
        # TODO, only works when no label is the prefix of another
        # Greedy matching
        matched_label = ''
        matched_index = len(output)
        for label in label_names:
          index = output.find(label)
          if index != -1 and index < matched_index:
            matched_label = label
            matched_index = index
        predicted_labels.append(matched_label)

      true_labels = test_df['label']
      if len(label_names) == 2:
        metrics0 = Metrics.cal_binary_metrics(true_labels, predicted_labels, label_names, pos_label)
      else:
        metrics0 = Metrics.cal_multiclass_metrics(true_labels, predicted_labels, label_names)
      metrics0['pos_label'] = pos_label
      predicted_labels = pd.Series(predicted_labels, index=test_df.index, name='predicted')
      save_state = pd.concat([predicted_labels, true_labels], axis=1)
      return metrics0, save_state


    def calc_confidence(label_names, candidate_seqs_all):
      # for each query, confidence: {'label 1': {probability} , 'label 2': {probability}}
      confidence_all = []
      for candidate_seqs in candidate_seqs_all:
          confidence = {label: 0 for label in label_names}
          for seq in candidate_seqs: # output label is the first token
              output_label_token = seq.text.strip().lower()
              if output_label_token in label_names:
                  confidence[output_label_token] += np.exp(seq.cumulative_logprob)
          softmax_probs = list(confidence.values()) / np.sum(list(confidence.values()))
          for (key, _), prob in zip(confidence.items(), softmax_probs):
              confidence[key] = prob
          confidence_all.append(confidence)
      return confidence_all


    output_text_all = []
    candidate_seqs_all = []
    # TODO: There can be cases where the outputs[0] is not a label name, but some of the left ones are. In that case, you consider the prediction as incorrect
    for output in outputs:
      output_text_all.append(output.outputs[0].text)
      candidate_seqs_all.append(output.outputs)

    metrics0, save_state = calc_metrics(output_text_all, test_df, label_names, pos_label)
    confidence_all = calc_confidence(label_names, candidate_seqs_all)
    return metrics0, confidence_all

outputs_file = os.path.join('/content/drive/MyDrive/LLM_ICL/Workspace/Result', f"{result_prefix}_outputs.pkl")
with open(outputs_file, 'wb') as f:
  pickle.dump(outputs, f)
logging.info(f"* Raw outputs are saved: {outputs_file}")
# Evaluate
metrics0, confidence = evaluate(
    outputs,
    task_info['label_names'],
    pos_label=task_info["positive_label"] if "positive_label" in task_info else None,
)
logging.info(f"* Finish predicting. Peformance = {metrics0}")


confidence_to_save_df = pd.concat([test_df['text'], pd.DataFrame(confidence)], axis=1)
confidence_file = os.path.join('/content/drive/MyDrive/LLM_ICL/Workspace/Result', f"{result_prefix}_confidence.csv")
confidence_to_save_df.to_csv(confidence_file, index=False)
logging.info(f"* Details of classification confidence are saved: {confidence_file}")

# Save results
result = {
    'exp': 'exp2_demos_label',
    'task': 'binary',
    'model': 'microsoft/Phi-3-small-128k-instruct',
    'random_seed': 42,
    'n_shots': n_shots,
    'retrieval': 'semantic',
    'metrics': metrics0,
}


with open(f'/content/drive/MyDrive/LLM_ICL/Workspace/Result/results_all.json', 'a') as file:
  file.write(json.dumps(result) + '\n')

logging.info(f"* Experiment records file updated: /content/drive/MyDrive/LLM_ICL/Workspace/Result/results_all.json")

## Phi-3-Mini (128K)

### Load the model

In [ ]:
model_phi3mini = LLM(model='microsoft/Phi-3-mini-128k-instruct', 
                     trust_remote_code=True, 
                     max_model_len=65536, 
                     enforce_eager=True,
                     tensor_parallel_size=2,
                     swap_space=8) # TO FIX [RuntimeError: Aborted due to the lack of CPU swap space. Please increase the swap space to avoid this error.]

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary', 'category']
shots = [0, 2, 4, 8, 16, 32, 64, 128]
retrievals = ['semantic', 'lexical', 'random']

for task in tasks:
  for shot in shots:
    if shot == 0:
      run_experiment(exp_name='exp1', task_name=task, model_name='phi3mini', n_shots=shot, model=model_phi3mini, retrieval='random')
      continue
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1', task_name=task, model_name='phi3mini', n_shots=shot, retrieval=retrieval, model=model_phi3mini, n_runs=10)
      else:
        run_experiment(exp_name='exp1', task_name=task, model_name='phi3mini', n_shots=shot, retrieval=retrieval, model=model_phi3mini)



## Qwen-2.5-7B (128K)

### Load the model

In [ ]:
model_qwen = LLM(model='Qwen/Qwen2.5-7B', trust_remote_code=True, tensor_parallel_size=2)

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary']
shots = [0, 2, 4, 8, 16, 32, 64, 128]
retrievals = ['semantic', 'lexical', 'random']

for task in tasks:
  for shot in shots:
    if shot == 0:
      run_experiment(exp_name='exp1', task_name=task, model_name='qwen', n_shots=shot, model=model_qwen, retrieval='random')
      continue
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1', task_name=task, model_name='qwen', n_shots=shot, retrieval=retrieval, model=model_qwen, n_runs=10)
      else:
        run_experiment(exp_name='exp1', task_name=task, model_name='qwen', n_shots=shot, retrieval=retrieval, model=model_qwen)


### Exp 1.2: Instruct vs Non-Instruct

This block checks whether instruction tuning improves binary detection quality under the same ICL pipeline.
We keep retrieval as `semantic` and compare model variants directly.

In [ ]:
# @title non-instruct version
model_qwen_noninstruct = LLM(model='Qwen/Qwen2.5-7B', trust_remote_code=True)

run_experiment(exp_name='exp1', task_name='binary', model_name='Qwen/Qwen2.5-7B', n_shots=32, model=model_qwen_noninstruct, retrieval='semantic')

### Exp 1.3: Demonstrations Without Labels

This ablation removes answer labels from demonstrations to measure how much explicit supervision contributes in-context.
The setup follows Exp 1 but sets `has_demos_label=False`.

In [ ]:
# w/o demos label

tasks = ['binary', 'category']
shots = [32]
retrievals = ['random', 'semantic']

for task in tasks:
  for shot in shots:
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='qwen', n_shots=shot, retrieval=retrieval, model=model_qwen, n_runs=3, has_demos_label=False)
      else:
        run_experiment(exp_name='exp2_demos_label', task_name=task, model_name='qwen', n_shots=shot, retrieval=retrieval, model=model_qwen, has_demos_label=False)

### Exp 7: Needle-in-Haystack Robustness

A sanity test where the query sample is inserted into demonstrations.
In principle, a robust ICL decision process should approach perfect recovery in this controlled setup.

In [ ]:
shots = [2, 4, 8, 16, 32, 64, 128]

for shot in shots:
    run_experiment(exp_name='exp7_haystack', task_name='binary', model_name='qwen', n_shots=shot, model=model_qwen, retrieval='random', needle_in_haystack=True)

## Gemma-2B (8K)

### Load the model

In [ ]:
model_gemma = LLM(model='google/gemma-2b', trust_remote_code=True, enable_prefix_caching=True)

### Exp 1.1: Search for the best ICL strategy

Test performance trends over K-shot settings and retrieval methods (`random`, `lexical`, `semantic`).

Notes: the random strategy is repeated with multiple runs to estimate variance.

In [ ]:
tasks = ['binary']
shots = [0, 2, 4, 8, 16, 32]
retrievals = ['random', 'lexical', 'semantic']

for task in tasks:
  for shot in shots:
    if shot == 0:
      run_experiment(exp_name='exp1', task_name=task, model_name='gemma', n_shots=shot, model=model_gemma, retrieval='random')
      continue
    for retrieval in retrievals:
      if retrieval == 'random':
        run_experiment(exp_name='exp1', task_name=task, model_name='gemma', n_shots=shot, retrieval=retrieval, model=model_gemma, n_runs=10)
      else:
        run_experiment(exp_name='exp1', task_name=task, model_name='gemma', n_shots=shot, retrieval=retrieval, model=model_gemma)

